In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

RESULTS_DIR = Path('../results')
PLOTS_DIR = Path('../results')
PLOTS_DIR.mkdir(exist_ok=True)

def load(filename):
    with open(RESULTS_DIR / f'{filename}.json') as f:
        return json.load(f)

sns.set_theme(style='whitegrid', font_scale=1.1)
COLORS = {
    'logreg': '#2563eb',
    'deberta': '#dc2626',
    'baseline': '#6b7280',
    'loss': '#16a34a',
    'confidence': '#ea580c',
    'heuristic': '#7c3aed',
    'all_data': '#0891b2',
    'human_only': '#be185d',
}
print('setup done')

## Plot 1 — ToxicChat Degradation Curves (PR-AUC)

In [ ]:
lr = load('noise_sweep_logreg_toxicchat')
db = load('noise_sweep_deberta_toxicchat')

noise_levels = [0.0, 0.1, 0.2, 0.4]
noise_labels = ['0%', '10%', '20%', '40%']

lr_prauc = [lr[str(n)]['prauc_mean'] for n in noise_levels]
lr_std   = [lr[str(n)]['prauc_std']  for n in noise_levels]
db_prauc = [db[str(n)]['prauc_mean'] for n in noise_levels]
db_std   = [db[str(n)]['prauc_std']  for n in noise_levels]

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(noise_labels, lr_prauc, 'o-', color=COLORS['logreg'], linewidth=2.5, markersize=8, label='LogReg')
ax.fill_between(noise_labels,
    [m - s for m, s in zip(lr_prauc, lr_std)],
    [m + s for m, s in zip(lr_prauc, lr_std)],
    alpha=0.15, color=COLORS['logreg'])

ax.plot(noise_labels, db_prauc, 's-', color=COLORS['deberta'], linewidth=2.5, markersize=8, label='DeBERTa-v3-base')
ax.fill_between(noise_labels,
    [m - s for m, s in zip(db_prauc, db_std)],
    [m + s for m, s in zip(db_prauc, db_std)],
    alpha=0.15, color=COLORS['deberta'])

ax.axvline(x='20%', color='#f59e0b', linestyle='--', linewidth=1.8, alpha=0.8, label='Tipping point (20%)')

ax.set_xlabel('Label Noise Level', fontsize=12)
ax.set_ylabel('PR-AUC', fontsize=12)
ax.set_title('ToxicChat — PR-AUC Degradation Under Pipeline Noise', fontsize=13, fontweight='bold')
ax.legend(framealpha=0.9)
ax.set_ylim(0, 1.0)

for i, (lv, dv, nl) in enumerate(zip(lr_prauc, db_prauc, noise_labels)):
    ax.annotate(f'{lv:.3f}', (nl, lv), textcoords='offset points', xytext=(-18, 8), fontsize=8.5, color=COLORS['logreg'])
    ax.annotate(f'{dv:.3f}', (nl, dv), textcoords='offset points', xytext=(6, 8), fontsize=8.5, color=COLORS['deberta'])

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'degradation_curves_toxicchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved degradation_curves_toxicchat.png')

## Plot 2 — Tipping Point (LogReg, fine-grained sweep)

In [ ]:
tp = load('tipping_point_toxicchat')

noise_levels_tp = tp['noise_levels']
prauc_means = [tp['results'][str(n)]['prauc_mean'] for n in noise_levels_tp]
prauc_stds  = [tp['results'][str(n)]['prauc_std']  for n in noise_levels_tp]
breakpoint  = tp['tipping_point']['noise_level']

left_x  = [p[0] for p in tp['piecewise_fit']['left']]
left_y  = [p[1] for p in tp['piecewise_fit']['left']]
right_x = [p[0] for p in tp['piecewise_fit']['right']]
right_y = [p[1] for p in tp['piecewise_fit']['right']]

noise_pct = [f'{int(n*100)}%' for n in noise_levels_tp]

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(noise_pct, prauc_means, 'o-', color=COLORS['logreg'], linewidth=2.5, markersize=8, label='LogReg PR-AUC', zorder=3)
ax.fill_between(noise_pct,
    [m - s for m, s in zip(prauc_means, prauc_stds)],
    [m + s for m, s in zip(prauc_means, prauc_stds)],
    alpha=0.15, color=COLORS['logreg'], label='±1 std')

left_idx  = [noise_pct.index(f'{int(x*100)}%') for x in left_x]
right_idx = [noise_pct.index(f'{int(x*100)}%') for x in right_x]
ax.plot([noise_pct[i] for i in left_idx],  left_y,  '--', color='#6b7280', linewidth=1.5, alpha=0.7)
ax.plot([noise_pct[i] for i in right_idx], right_y, '--', color='#6b7280', linewidth=1.5, alpha=0.7, label='Piecewise fit')

bp_label = f'{int(breakpoint*100)}%'
ax.axvline(x=bp_label, color='#f59e0b', linestyle='--', linewidth=2, alpha=0.9)
ax.annotate(f'Tipping point\n{bp_label} noise',
    xy=(bp_label, tp['tipping_point']['prauc_mean']),
    xytext=(30, -40), textcoords='offset points',
    fontsize=9.5, color='#b45309',
    arrowprops=dict(arrowstyle='->', color='#b45309', lw=1.5))

ax.set_xlabel('Label Noise Level', fontsize=12)
ax.set_ylabel('PR-AUC', fontsize=12)
ax.set_title('Tipping Point — Where LogReg Starts Breaking (ToxicChat)', fontsize=13, fontweight='bold')
ax.legend(framealpha=0.9)
ax.set_ylim(0, 0.75)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'tipping_point_toxicchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved tipping_point_toxicchat.png')

## Plot 3 — Cleaning Recovery (LogReg)

In [ ]:
cl_lr = load('cleaning_logreg_toxicchat')

noise_levels_cl = ['0.1', '0.2', '0.4']
noise_labels_cl = ['10%', '20%', '40%']
strategies = ['noisy_baseline', 'loss', 'heuristic', 'confidence']
strategy_labels = ['Noisy baseline', 'Loss filter', 'Heuristic filter', 'Confidence filter']
strat_colors = [COLORS['baseline'], COLORS['loss'], COLORS['heuristic'], COLORS['confidence']]

x = np.arange(len(noise_labels_cl))
width = 0.2

fig, ax = plt.subplots(figsize=(9, 5))

for i, (strat, label, color) in enumerate(zip(strategies, strategy_labels, strat_colors)):
    means = [cl_lr[n][strat]['prauc_mean'] for n in noise_levels_cl]
    stds  = [cl_lr[n][strat]['prauc_std']  for n in noise_levels_cl]
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, means, width, label=label, color=color, alpha=0.85)
    ax.errorbar(x + offset, means, yerr=stds, fmt='none', color='#1f2937', capsize=3, linewidth=1.2)

ax.set_xlabel('Noise Level', fontsize=12)
ax.set_ylabel('PR-AUC', fontsize=12)
ax.set_title('Cleaning Recovery — LogReg on ToxicChat', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(noise_labels_cl)
ax.legend(framealpha=0.9)
ax.set_ylim(0, 0.75)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cleaning_recovery_logreg_toxicchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved cleaning_recovery_logreg_toxicchat.png')

## Plot 4 — Cleaning Recovery (DeBERTa)

In [ ]:
cl_db = load('cleaning_deberta_toxicchat')

fig, ax = plt.subplots(figsize=(9, 5))

for i, (strat, label, color) in enumerate(zip(strategies, strategy_labels, strat_colors)):
    means = [cl_db[n][strat]['prauc_mean'] for n in noise_levels_cl]
    stds  = [cl_db[n][strat]['prauc_std']  for n in noise_levels_cl]
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, means, width, label=label, color=color, alpha=0.85)
    ax.errorbar(x + offset, means, yerr=stds, fmt='none', color='#1f2937', capsize=3, linewidth=1.2)

ax.set_xlabel('Noise Level', fontsize=12)
ax.set_ylabel('PR-AUC', fontsize=12)
ax.set_title('Cleaning Recovery — DeBERTa-v3-base on ToxicChat', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(noise_labels_cl)
ax.legend(framealpha=0.9)
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cleaning_recovery_deberta_toxicchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved cleaning_recovery_deberta_toxicchat.png')

## Plot 5 — Quantity vs Quality

In [ ]:
qvq_lr = load('quantity_vs_quality_logreg_toxicchat')
qvq_db = load('quantity_vs_quality_deberta_toxicchat')

scenarios = ['All data\n(weak labels included)', 'Human annotation\nonly']
lr_means = [qvq_lr['all_data']['prauc_mean'], qvq_lr['human_only']['prauc_mean']]
lr_stds  = [qvq_lr['all_data']['prauc_std'],  qvq_lr['human_only']['prauc_std']]
db_means = [qvq_db['all_data']['prauc_mean'], qvq_db['human_only']['prauc_mean']]
db_stds  = [qvq_db['all_data']['prauc_std'],  qvq_db['human_only']['prauc_std']]

x = np.arange(len(scenarios))
width = 0.32

fig, ax = plt.subplots(figsize=(8, 5))

b1 = ax.bar(x - width/2, lr_means, width, label='LogReg', color=COLORS['logreg'], alpha=0.85)
ax.errorbar(x - width/2, lr_means, yerr=lr_stds, fmt='none', color='#1f2937', capsize=4, linewidth=1.4)

b2 = ax.bar(x + width/2, db_means, width, label='DeBERTa-v3-base', color=COLORS['deberta'], alpha=0.85)
ax.errorbar(x + width/2, db_means, yerr=db_stds, fmt='none', color='#1f2937', capsize=4, linewidth=1.4)

for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.3f}',
            xy=(bar.get_x() + bar.get_width() / 2, h),
            xytext=(0, 4), textcoords='offset points',
            ha='center', fontsize=9)

ax.set_ylabel('PR-AUC', fontsize=12)
ax.set_title('Quantity vs Quality — All Data vs Human Annotation Only (ToxicChat)', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=11)
ax.legend(framealpha=0.9)
ax.set_ylim(0, 1.0)

n_all   = '~4,300 samples'
n_human = '~2,380 samples'
ax.text(0, 0.03, n_all,   ha='center', fontsize=8.5, color='#6b7280')
ax.text(1, 0.03, n_human, ha='center', fontsize=8.5, color='#6b7280')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'quantity_vs_quality_toxicchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved quantity_vs_quality_toxicchat.png')

## Plot 6 — SST-2 vs ToxicChat: Synthetic vs Real Pipeline Noise

In [ ]:
sst2_lr = load('noise_sweep_logreg')
tc_lr   = load('noise_sweep_logreg_toxicchat')

noise_levels_shared = [0.0, 0.1, 0.2, 0.4]
noise_labels_shared = ['0%', '10%', '20%', '40%']

sst2_f1   = [sst2_lr[str(n)]['f1_mean']        for n in noise_levels_shared]
sst2_std  = [sst2_lr[str(n)]['f1_std']          for n in noise_levels_shared]
tc_prauc  = [tc_lr[str(n)]['prauc_mean']        for n in noise_levels_shared]
tc_std    = [tc_lr[str(n)]['prauc_std']          for n in noise_levels_shared]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(noise_labels_shared, sst2_f1, 'o-', color='#0891b2', linewidth=2.5, markersize=8)
ax.fill_between(noise_labels_shared,
    [m - s for m, s in zip(sst2_f1, sst2_std)],
    [m + s for m, s in zip(sst2_f1, sst2_std)],
    alpha=0.15, color='#0891b2')
ax.set_title('SST-2 — Synthetic Noise\n(LogReg, F1)', fontsize=12, fontweight='bold')
ax.set_xlabel('Noise Level')
ax.set_ylabel('F1')
ax.set_ylim(0.6, 1.0)

ax = axes[1]
ax.plot(noise_labels_shared, tc_prauc, 'o-', color=COLORS['logreg'], linewidth=2.5, markersize=8)
ax.fill_between(noise_labels_shared,
    [m - s for m, s in zip(tc_prauc, tc_std)],
    [m + s for m, s in zip(tc_prauc, tc_std)],
    alpha=0.15, color=COLORS['logreg'])
ax.axvline(x='20%', color='#f59e0b', linestyle='--', linewidth=1.8, alpha=0.8, label='Tipping point')
ax.set_title('ToxicChat — Real Pipeline Noise\n(LogReg, PR-AUC)', fontsize=12, fontweight='bold')
ax.set_xlabel('Noise Level')
ax.set_ylabel('PR-AUC')
ax.set_ylim(0, 0.75)
ax.legend()

fig.suptitle('Synthetic Noise (SST-2) vs Real Pipeline Noise (ToxicChat) — LogReg', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'sst2_vs_toxicchat_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved sst2_vs_toxicchat_comparison.png')

## Plot 7 — F1 Trap: PR-AUC vs F1 Macro divergence

In [ ]:
tp_data = load('tipping_point_toxicchat')

noise_levels_tp = tp_data['noise_levels']
prauc_vals = [tp_data['results'][str(n)]['prauc_mean'] for n in noise_levels_tp]
f1_vals    = [tp_data['results'][str(n)]['f1_macro_mean'] for n in noise_levels_tp]
noise_pct  = [f'{int(n*100)}%' for n in noise_levels_tp]

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(noise_pct, prauc_vals, 'o-', color=COLORS['deberta'], linewidth=2.5, markersize=8, label='PR-AUC (primary metric)')
ax.plot(noise_pct, f1_vals,   's--', color=COLORS['logreg'],  linewidth=2.5, markersize=8, label='F1 Macro')

ax.axvline(x='20%', color='#f59e0b', linestyle=':', linewidth=2, alpha=0.8)
ax.annotate('Both metrics agree\nuntil here', xy=('15%', 0.62), fontsize=9, color='#374151',
    ha='center',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#fef9c3', edgecolor='#f59e0b', alpha=0.8))
ax.annotate('F1 keeps rising\nPR-AUC collapses', xy=('27%', 0.56), fontsize=9, color='#374151',
    ha='center',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#fee2e2', edgecolor='#dc2626', alpha=0.8))

ax.set_xlabel('Label Noise Level', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('The F1 Trap — F1 Macro Masks Degradation on Imbalanced Data', fontsize=13, fontweight='bold')
ax.legend(framealpha=0.9)
ax.set_ylim(0.1, 0.8)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'f1_trap_toxicchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved f1_trap_toxicchat.png')

## All plots saved to results/

In [ ]:
plots = list(PLOTS_DIR.glob('*.png'))
print(f'{len(plots)} plots saved:')
for p in sorted(plots):
    print(f'  {p.name}')